# Vector Stores and Semantic Search

For this activity, a basic vector store was created from scratch to search through documents using natural language queries. The project uses sentence-transformers to generate embeddings and numpy to compare similarities with cosine similarity, without using frameworks such as FAISS or LangChain.

In [1]:
from sentence_transformers import SentenceTransformer

## Part I: Basic Vector Store Implementation

### Data Classes

The notebook includes two data classes: Document,which stores the text and its metadata, and SearchResult,which contains the similarity score and the matching document.

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

### VectorStore

The VectorStore class stores both the documents and their embeddings. when add_documents is used, the text is converted into vectors with the embedding model. During a search, the query is also transformed into a vector and compared with the stored embeddings using cosine similarity,returning the most similar documents ordered by score

In [3]:
import numpy as np

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeds = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

        if self.embeddings is None:
            self.embeddings = new_embeds
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeds])

        self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embed = self.model.encode([query], normalize_embeddings=True)

        scores = np.dot(self.embeddings, query_embed.T).flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(score=float(scores[idx]), document=self.documents[idx]))

        return results

### Loading the Animal Fun Facts Dataset

The CSV file is loaded with pandas,and each row is converted into a Document.the text column is used as the main content, while the other columns are saved as metadata.Rows without text are removed since they can’t be converted into embeddings.

In [4]:
import pandas as pd

df = pd.read_csv("../data/animal-fun-facts-dataset.csv")
df = df.dropna(subset=["text"])

print(f"Total rows: {len(df)}")
print(f"Unique animals: {df['animal_name'].nunique()}")
print(f"Columns: {list(df.columns)}")
df.head()

Total rows: 7731
Unique animals: 2500
Columns: ['animal_name', 'source', 'text', 'media_link', 'wikipedia_link']


,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [5]:
animal_docs = []
for _, row in df.iterrows():
    doc = Document(
        text=row["text"],
        metadata={
            "animal_name": str(row["animal_name"]),
            "source": str(row["source"]),
            "media_link": str(row.get("media_link", "")),
            "wikipedia_link": str(row.get("wikipedia_link", "")),
        }
    )
    animal_docs.append(doc)

print(f"Created {len(animal_docs)} documents")
print(f"Sample text: {animal_docs[0].text[:100]}...")
print(f"Sample metadata: {animal_docs[0].metadata}")

Created 7731 documents
Sample text: Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"...
Sample metadata: {'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': 'nan', 'wikipedia_link': '/wiki/Aardvark'}


### Building the Vector Store

The all-MiniLM-L6-v2 model is used to generate the embeddings.It is a lightweight model that works well for smantic searches, although processing all the documents can take a few minutes.

In [6]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

store = VectorStore(embed_model)
store.add_documents(animal_docs)

print(f"Documents: {len(store.documents)}")
print(f"Embedding shape: {store.embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Documents: 7731
Embedding shape: (7731, 384)


### Querying the Vector Store

A helper function is created to run searches and display the results in a cleaner and easier to read format,including the similarity score,matched text,and animal name.

In [7]:
def show_results(query, results):
    print(f'Query: "{query}"')
    print("-" * 80)
    for i, r in enumerate(results):
        print(f"  {i+1}. [score: {r.score:.4f}] ({r.document.metadata['animal_name']})")
        print(f"     {r.document.text[:150]}")
        print()

Five different queries are tested to check how well the semantic search understands the meaning of the text instead of only matching exact words.

In [8]:
q1 = "animals that can fly"
results1 = store.search(q1, top_k=5)
show_results(q1, results1)

Query: "animals that can fly"
--------------------------------------------------------------------------------
  1. [score: 0.7076] (colugo (flying lemur))
     They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.

  2. [score: 0.6901] (bird)
     Not all birds are able to fly!

  3. [score: 0.6481] (secretary bird)
     They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.

  4. [score: 0.6437] (bat)
     Bats are the only mammals with wings, and the only ones that can truly fly

  5. [score: 0.6423] (flamingo)
     Flamingos can fly!.
Flamingos kept in zoo’s have often had their wings clipped. In the wild, flamingos can fly, and they use their wings to migrate to



In [9]:
q2 = "fastest land animal"
results2 = store.search(q2, top_k=5)
show_results(q2, results2)

Query: "fastest land animal"
--------------------------------------------------------------------------------
  1. [score: 0.8639] (cheetah)
     The fastest land mammal in the world!

  2. [score: 0.8303] (peregrine falcon)
     Fastest animal on Earth

  3. [score: 0.7082] (white rhinoceros)
     The second largest animal on the land!

  4. [score: 0.7077] (falcon)
     The fastest creatures on the planet!

  5. [score: 0.6880] (snap-jaw ant)
     They’re one of the fastest animals alive.
It may seem weird to picture an ant at close to 400kmph (248mph), but this ant isn’t running this fast, so t



In [10]:
q3 = "animals that live in the ocean"
results3 = store.search(q3, top_k=5)
show_results(q3, results3)

Query: "animals that live in the ocean"
--------------------------------------------------------------------------------
  1. [score: 0.6498] (vaquita)
     Smallest cetacean in the ocean

  2. [score: 0.6452] (white shark)
     White Sharks live in all of the world's oceans.

  3. [score: 0.6317] (bonito fish)
     May eat squid or other small invertebrate ocean life

  4. [score: 0.6217] (sharks)
     Sharks live all over the world, from warm, tropical lagoons to polar seas. Some even inhabit freshwater lakes and rivers!

  5. [score: 0.6217] (sea roach)
     They breathe through gills but live on land



In [11]:
q4 = "poisonous or venomous creatures"
results4 = store.search(q4, top_k=5)
show_results(q4, results4)

Query: "poisonous or venomous creatures"
--------------------------------------------------------------------------------
  1. [score: 0.7251] (pufferfish)
     The second most poisonous creature in the world!

  2. [score: 0.7204] (poison dart frog)
     They are poisonous, not venomous..
This means that they do not inject their toxins into others, like snakes, they instead have to be consumed or licke

  3. [score: 0.7149] (millipede)
     Some species have a poisonous bite!

  4. [score: 0.7010] (stonefish)
     The most venomous fish in the world

  5. [score: 0.6917] (taipan)
     The Most Venomous Snakes On Earth



In [12]:
q5 = "animals with unusual sleeping habits"
results5 = store.search(q5, top_k=5)
show_results(q5, results5)

Query: "animals with unusual sleeping habits"
--------------------------------------------------------------------------------
  1. [score: 0.6862] (coatimundi)
     These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of food, grooming, and resting.

  2. [score: 0.6447] (syrian hamster)
     They sleep during the day and are awake at night

  3. [score: 0.5910] (african lion)
     African lions sleep - or doze, rather- about 20 hours a day

  4. [score: 0.5844] (elephant)
     Elephants can sleep standing up and only need a couple of hours of sleep each day..
But not always, they do lay down every few nights to sleep.

  5. [score: 0.5837] (jerboa)
     They are nocturnal to help them escape high temperatures..
They spend all day in burrows they make in the sand, only coming out at night when it is co



The results show that the vector store can find related documents based on meaning, even when the exact words are different. For example, a search like “fastest land animal” still returns documents about very fast animals, even if those exact words are not used.

## Part II: Filtered Vector Store

### FilteredVectorStore

This works similarly to the basic VectorStore, but it also allows filtering by metadata. When a metadata_filter is provided, the search only looks through documents that match those values before returning the top results.

In [13]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeds = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

        if self.embeddings is None:
            self.embeddings = new_embeds
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeds])

        self.documents.extend(documents)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embed = self.model.encode([query], normalize_embeddings=True)

        scores = np.dot(self.embeddings, query_embed.T).flatten()

        candidates = []
        for i, score in enumerate(scores):
            if metadata_filter is not None:
                match = all(
                    self.documents[i].metadata.get(k) == v
                    for k, v in metadata_filter.items()
                )
                if not match:
                    continue
            candidates.append((i, score))

        candidates.sort(key=lambda x: x[1], reverse=True)
        candidates = candidates[:top_k]

        results = []
        for idx, score in candidates:
            results.append(SearchResult(score=float(score), document=self.documents[idx]))

        return results

### Selecting a Dataset for FilteredVectorStore

For this part, the [Consumer Reviews of Amazon Products](https://www.kaggle.com/datasets/datafiniti/consumer-reviews-of-amazon-products) dataset is used. It includes around 28,000 reviews of Amazon products like Kindle, Fire TV, and Echo, along with metadata such as the product name,brand, and star rating. A random sample of 500 reviews was used to make the encoding process faster. The metadata fields used for filtering are brand, rating, and product_name.

In [14]:
reviews_df = pd.read_csv("../data/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv")

reviews_df = reviews_df.dropna(subset=["reviews.text"])
reviews_df = reviews_df[reviews_df["reviews.text"].str.strip() != ""]

print(f"Total reviews: {len(reviews_df)}")
print(f"Columns: {list(reviews_df.columns)}")
reviews_df[["name", "brand", "reviews.rating", "reviews.text"]].head()

Total reviews: 28332
Columns: ['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs']


,name,brand,reviews.rating,reviews.text
0,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,3,I order 3 of them and one of the item is bad q...
1,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,4,Bulk is always the less expensive way to go fo...
2,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,5,Well they are not Duracell but for the price i...
3,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,5,Seem to work as well as name brand batteries a...
4,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,5,These batteries are very long lasting the pric...


In [15]:
sample_df = reviews_df.sample(n=500, random_state=42)

review_docs = []
for _, row in sample_df.iterrows():
    doc = Document(
        text=str(row["reviews.text"]),
        metadata={
            "brand": str(row.get("brand", "unknown")),
            "rating": str(int(row["reviews.rating"])),
            "product_name": str(row.get("name", "unknown")),
        }
    )
    review_docs.append(doc)

print(f"Created {len(review_docs)} review documents")
print(f"Sample text: {review_docs[0].text[:120]}...")
print(f"Sample metadata: {review_docs[0].metadata}")

brand_counts = {}
for d in review_docs:
    b = d.metadata["brand"]
    brand_counts[b] = brand_counts.get(b, 0) + 1
print(f"\nBrands: {brand_counts}")

Created 500 review documents
Sample text: Awesome tablet. I was amazed how fast it is. And the software is very user friendly...
Sample metadata: {'brand': 'Amazon', 'rating': '5', 'product_name': 'All-New Fire HD 8 Kids Edition Tablet, 8 HD Display, 32 GB, Blue Kid-Proof Case'}

Brands: {'Amazon': 319, 'Amazonbasics': 181}


In [16]:
filtered_store = FilteredVectorStore(embed_model)
filtered_store.add_documents(review_docs)

print(f"Documents in filtered store: {len(filtered_store.documents)}")
print(f"Embedding shape: {filtered_store.embeddings.shape}")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Documents in filtered store: 500
Embedding shape: (500, 384)


### Querying with Metadata Filters

A similar helper function is used here,but this time it also shows tje metadata of each result. Five example queries are tested below using different metadata filters to make the search more specific.

In [17]:
def show_filtered_results(query, results, filter_used):
    print(f'Query: "{query}"')
    print(f"Filter: {filter_used}")
    print("-" * 80)
    for i, r in enumerate(results):
        print(f"  {i+1}. [score: {r.score:.4f}] {r.document.text[:120]}")
        print(f"     metadata: {r.document.metadata}")
        print()

In [18]:
# query 1:only 5-star reviews
fq1 = "amazing product love it"
f1 = {"rating": "5"}
fr1 = filtered_store.search(fq1, top_k=3, metadata_filter=f1)
show_filtered_results(fq1, fr1, f1)

Query: "amazing product love it"
Filter: {'rating': '5'}
--------------------------------------------------------------------------------
  1. [score: 0.7010] good product
     metadata: {'brand': 'Amazonbasics', 'rating': '5', 'product_name': 'AmazonBasics AAA Performance Alkaline Batteries (36 Count)'}

  2. [score: 0.6976] Great Product and Price
     metadata: {'brand': 'Amazonbasics', 'rating': '5', 'product_name': 'AmazonBasics AAA Performance Alkaline Batteries (36 Count)'}

  3. [score: 0.6310] Great product and works great.
     metadata: {'brand': 'Amazonbasics', 'rating': '5', 'product_name': 'AmazonBasics AAA Performance Alkaline Batteries (36 Count)'}



In [19]:
# query 2:negative reviews(1 star)
fq2 = "terrible quality stopped working"
f2 = {"rating": "1"}
fr2 = filtered_store.search(fq2, top_k=3, metadata_filter=f2)
show_filtered_results(fq2, fr2, f2)

Query: "terrible quality stopped working"
Filter: {'rating': '1'}
--------------------------------------------------------------------------------
  1. [score: 0.3578] The screen breaks way to easy it drop one time broke
     metadata: {'brand': 'Amazon', 'rating': '1', 'product_name': 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Pink Kid-Proof Case'}

  2. [score: 0.3219] These batteries are awful, have no power and die fast
     metadata: {'brand': 'Amazonbasics', 'rating': '1', 'product_name': 'AmazonBasics AAA Performance Alkaline Batteries (36 Count)'}

  3. [score: 0.3082] These batteries are junk. Some of the new once have no more power than the once they replaced. If they are good, they wa
     metadata: {'brand': 'Amazonbasics', 'rating': '1', 'product_name': 'AmazonBasics AA Performance Alkaline Batteries (48 Count) - Packaging May Vary'}



In [20]:
# query 3:Amazon brand+4-star reviews
fq3 = "good sound quality speaker"
f3 = {"brand": "Amazon", "rating": "4"}
fr3 = filtered_store.search(fq3, top_k=3, metadata_filter=f3)
show_filtered_results(fq3, fr3, f3)

Query: "good sound quality speaker"
Filter: {'brand': 'Amazon', 'rating': '4'}
--------------------------------------------------------------------------------
  1. [score: 0.4699] Good quality product. Clear display and easy to use.
     metadata: {'brand': 'Amazon', 'rating': '4', 'product_name': 'All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Black'}

  2. [score: 0.3970] I'm happy with my purchase is all I can say. The only complaint I have is that the 3.5 mm Jack is right next to the volu
     metadata: {'brand': 'Amazon', 'rating': '4', 'product_name': 'Fire HD 8 Tablet with Alexa, 8 HD Display, 16 GB, Tangerine - with Special Offers'}

  3. [score: 0.3540] Gives a good quality, being that it's a poloroid camera
     metadata: {'brand': 'Amazon', 'rating': '4', 'product_name': 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Pink Kid-Proof Case'}



In [21]:
# query 4:battery life concerns across all ratings
fq4 = "battery life charging"
f4 = {"brand": "Amazon"}
fr4 = filtered_store.search(fq4, top_k=3, metadata_filter=f4)
show_filtered_results(fq4, fr4, f4)

Query: "battery life charging"
Filter: {'brand': 'Amazon'}
--------------------------------------------------------------------------------
  1. [score: 0.4445] When this item is charged it works great, but it takes too long to charge. Asking my 3 year old to wait about 3 hours (n
     metadata: {'brand': 'Amazon', 'rating': '2', 'product_name': 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Pink Kid-Proof Case'}

  2. [score: 0.4244] Battery life is unreal! So much faster than my old kindle.
     metadata: {'brand': 'Amazon', 'rating': '5', 'product_name': 'Kindle Oasis E-reader with Leather Charging Cover - Walnut, 6 High-Resolution Display (300 ppi), Wi-Fi - Includes Special Offers'}

  3. [score: 0.4037] My old Kindle was not working so good anymore so bought this one on a Black Friday Deal from Best Buy. Price was great. 
     metadata: {'brand': 'Amazon', 'rating': '4', 'product_name': 'All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Black'}



In [22]:
# query 5:3-star reviews (mixed opinions)
fq5 = "decent but has issues"
f5 = {"rating": "3"}
fr5 = filtered_store.search(fq5, top_k=3, metadata_filter=f5)
show_filtered_results(fq5, fr5, f5)

Query: "decent but has issues"
Filter: {'rating': '3'}
--------------------------------------------------------------------------------
  1. [score: 0.3895] This is decent reader and Netflix device. Lots of bugs and gotchas when trying to use - icons going off screen, random l
     metadata: {'brand': 'Amazon', 'rating': '3', 'product_name': 'All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Black'}

  2. [score: 0.3513] Overall it is a good tablet, but I am not a fan of the limitations of the silk browser and Amazon app store.
     metadata: {'brand': 'Amazon', 'rating': '3', 'product_name': 'Fire Tablet with Alexa, 7 Display, 16 GB, Blue - with Special Offers'}

  3. [score: 0.3109] I bought it and wrapped it up immediately. So I couldn't say either way if it was good or not.
     metadata: {'brand': 'Amazon', 'rating': '3', 'product_name': 'Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Pink Kid-Proof Case'}



The filtered results show that the metadata filtering works as expected.For example, when using rating: 1,the search only returns low-rated reviews. Combining filters like brand: Amazon and rating: 4 makes the results even more specific, which can be useful for focused product analysis.

## REST API with FilteredVectorStore

A REST API built with Flask is implemented below using the FilteredVectorStore.The API allows creating documents, retrieving them by ID, and performing semantic searches with optional metadata filters. The documents are stored as articles and include three required metadata fields: category, author, and year.

In [23]:
from flask import Flask, jsonify, request as flask_request
import uuid

### Chunking

If a document is longer than 500 characters, it is automatically divided into smaller chunks.Each chunk keeps the original metadata and alsoincludes an article_id so it can still be linked back to the main article.

In [24]:
CHUNK_SIZE = 400
MAX_DOC_LEN = 500

def chunk_text(text):
    if len(text) <= MAX_DOC_LEN:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunks.append(text[start:end])
        start = end
    return chunks

short = "This is a short text."
long = "x" * 850
print(f"Short text: {len(chunk_text(short))} chunk(s)")
print(f"Long text: {len(chunk_text(long))} chunk(s), sizes: {[len(c) for c in chunk_text(long)]}")

Short text: 1 chunk(s)
Long text: 3 chunk(s), sizes: [400, 400, 50]


### Flask App and Routes

The app includes three main routes:

-POST /articles — creates and stores a new article, validating the metadata and splitting long texts into chunks.

-GET /articles/<id> — retrieves the complete original article.

-POST /articles/search — performs a semantic search with optional metadata filters and returns the matching chunks with their scores and metadata.

In [25]:
app = Flask(__name__)

REQUIRED_META = {"category", "author", "year"}

api_store = FilteredVectorStore(embed_model)
full_articles = {}  

@app.route("/articles", methods=["POST"])
def create_article():
    data = flask_request.get_json()
    if not data or "text" not in data or "metadata" not in data:
        return jsonify({"error": "request must include text and metadata"}), 400

    meta = data["metadata"]
    if not isinstance(meta, dict) or set(meta.keys()) != REQUIRED_META:
        return jsonify({"error": f"metadata must contain exactly: {sorted(REQUIRED_META)}"}), 400

    for key in REQUIRED_META:
        if not isinstance(meta[key], str) or not meta[key].strip():
            return jsonify({"error": f"metadata field '{key}' must be a non-empty string"}), 400

    article_id = str(uuid.uuid4())[:8]
    full_articles[article_id] = {"id": article_id, "text": data["text"], "metadata": meta}

    chunks = chunk_text(data["text"])
    chunk_docs = []
    for chunk in chunks:
        chunk_meta = {**meta, "article_id": article_id}
        chunk_docs.append(Document(text=chunk, metadata=chunk_meta))
    api_store.add_documents(chunk_docs)

    return jsonify({"id": article_id, "chunks_created": len(chunks), "message": "article created"}), 201


@app.route("/articles/<article_id>", methods=["GET"])
def get_article(article_id):
    article = full_articles.get(article_id)
    if not article:
        return jsonify({"error": "article not found"}), 404
    return jsonify(article), 200


@app.route("/articles/search", methods=["POST"])
def search_articles():
    data = flask_request.get_json()
    if not data or "query" not in data:
        return jsonify({"error": "request must include a query field"}), 400

    query = data["query"]
    top_k = data.get("top_k", 5)
    meta_filter = data.get("metadata_filter", None)

    results = api_store.search(query, top_k=top_k, metadata_filter=meta_filter)
    response = []
    for r in results:
        response.append({
            "similarity": f"{r.score * 100:.1f}%",
            "score": round(r.score, 4),
            "text": r.document.text,
            "metadata": r.document.metadata,
        })

    return jsonify({"query": query, "results": response}), 200


print("Flask app defined with 3 routes: POST /articles, GET /articles/<id>, POST /articles/search")

Flask app defined with 3 routes: POST /articles, GET /articles/<id>, POST /articles/search


### Testing the API

Flask’s built-in test client is used to simulate HTTP requests directly in the notebook,allowing everything to run in one place without needing a separate terminal.

In [26]:
import json as js

client = app.test_client()

short_review = reviews_df[reviews_df["reviews.text"].str.len() <= 500].iloc[0]
long_review = reviews_df[reviews_df["reviews.text"].str.len() > 500].iloc[0]

print("Creating short article (from dataset, no chunking)...")
resp = client.post("/articles", json={
    "text": short_review["reviews.text"],
    "metadata": {"category": "electronics", "author": str(short_review.get("reviews.username", "anonymous")), "year": "2019"}
})
short_id = resp.get_json()["id"]
print(f"  Status: {resp.status_code} | {resp.get_json()}")

print(f"\nCreating long article ({len(long_review['reviews.text'])} chars, should chunk)...")
resp = client.post("/articles", json={
    "text": long_review["reviews.text"],
    "metadata": {"category": "electronics", "author": str(long_review.get("reviews.username", "anonymous")), "year": "2019"}
})
long_id = resp.get_json()["id"]
print(f"  Status: {resp.status_code} | {resp.get_json()}")

print("\nCreating article with missing metadata field...")
resp = client.post("/articles", json={
    "text": "This should fail.",
    "metadata": {"category": "ai", "author": "Test"}
})
print(f"  Status: {resp.status_code} | {resp.get_json()}")

extra_samples = reviews_df.sample(n=3, random_state=99)
print("\nAdding 3 more articles from the dataset...")
for _, row in extra_samples.iterrows():
    r = client.post("/articles", json={
        "text": str(row["reviews.text"]),
        "metadata": {"category": "electronics", "author": str(row.get("reviews.username", "anonymous")), "year": "2019"}
    })
    print(f"  {r.get_json()}")

Creating short article (from dataset, no chunking)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Status: 201 | {'chunks_created': 1, 'id': 'cdc0e66c', 'message': 'article created'}

Creating long article (4883 chars, should chunk)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Status: 201 | {'chunks_created': 13, 'id': '9bba0a1d', 'message': 'article created'}

Creating article with missing metadata field...
  Status: 400 | {'error': "metadata must contain exactly: ['author', 'category', 'year']"}

Adding 3 more articles from the dataset...


...

  {'chunks_created': 1, 'id': '91b7974b', 'message': 'article created'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  {'chunks_created': 1, 'id': 'b160bdc8', 'message': 'article created'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  {'chunks_created': 1, 'id': '3f47989e', 'message': 'article created'}


In [27]:
print("GET /articles/" + short_id)
resp = client.get(f"/articles/{short_id}")
print(f"Status: {resp.status_code}")
print(js.dumps(resp.get_json(), indent=2))

print("\nGET /articles/doesnotexist")
resp = client.get("/articles/doesnotexist")
print(f"Status: {resp.status_code} | {resp.get_json()}")

GET /articles/cdc0e66c
Status: 200
{
  "id": "cdc0e66c",
  "metadata": {
    "author": "Byger yang",
    "category": "electronics",
    "year": "2019"
  },
  "text": "I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pcs of aluminum to make the battery work."
}

GET /articles/doesnotexist
Status: 404 | {'error': 'article not found'}


In [28]:
print("Search: 'great sound quality' (no filter)")
resp = client.post("/articles/search", json={"query": "great sound quality", "top_k": 3})
print(js.dumps(resp.get_json(), indent=2))

Search: 'great sound quality' (no filter)
{
  "query": "great sound quality",
  "results": [
    {
      "metadata": {
        "article_id": "b160bdc8",
        "author": "jwilburn",
        "category": "electronics",
        "year": "2019"
      },
      "score": 0.333,
      "similarity": "33.3%",
      "text": "Bought for my son. Works great. Great tablet for the $."
    },
    {
      "metadata": {
        "article_id": "cdc0e66c",
        "author": "Byger yang",
        "category": "electronics",
        "year": "2019"
      },
      "score": 0.2311,
      "similarity": "23.1%",
      "text": "I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pcs of aluminum to make the battery work."
    },
    {
      "metadata": {
        "article_id": "9bba0a1d",
        "author": "ByC",
        "category": "electronics",
        "year": "2019"
      },
      "score": 0.1822,
      "similarity": "18.2%",
      "text": "r the cost (0.31/ea in a 36

In [29]:
print("Search: 'battery life' (filter: category=electronics)")
resp = client.post("/articles/search", json={
    "query": "battery life lasts long",
    "top_k": 3,
    "metadata_filter": {"category": "electronics"}
})
result = resp.get_json()
print(js.dumps(result, indent=2))

if result["results"]:
    parent_id = result["results"][0]["metadata"].get("article_id")
    if parent_id:
        full = client.get(f"/articles/{parent_id}").get_json()
        print(f"\nChunk belongs to article_id: {parent_id}")
        print(f"Full article length: {len(full['text'])} chars")
        print(f"Preview: {full['text'][:120]}...")

Search: 'battery life' (filter: category=electronics)
{
  "query": "battery life lasts long",
  "results": [
    {
      "metadata": {
        "article_id": "9bba0a1d",
        "author": "ByC",
        "category": "electronics",
        "year": "2019"
      },
      "score": 0.3705,
      "similarity": "37.1%",
      "text": "time100 mA 1005 mAh 10 hrs200 mA 864 mAh 4.3 hrs400 mA 670 mAh 1.7 hrsAs you can see, the Amazon batteries were very comperable ... but, at the current time the AmazonBasics batteries are 0.31/ea (36 pack), while the ACDelco batteries are about 0.21/ea (48 pack) -- so the ACDelco are significantly cheaper per mAh. I did not test shelf life, so it's possible that these may hold up better sitting in"
    },
    {
      "metadata": {
        "article_id": "9bba0a1d",
        "author": "ByC",
        "category": "electronics",
        "year": "2019"
      },
      "score": 0.3123,
      "similarity": "31.2%",
      "text": "AmazonBasics batteries are quite good in ter

The tests confirm that incomplete metadata is rejected, long documents are properlysplit into chunks while keeping their article_id connection,and filtered searches return more precise results

## Personal Reflection
In conclusion, this activity helped me better understand how vector databases like FAISS or Pinecone actually work behind the scenes. Implementing cosine similarity with numpy made the main idea feel much simpler: convert text into vectors and compare how similar they are. It also made me realize that the real challenge in larger systems is handling huge amounts of data efficiently.

Something I found interesting was how well all-MiniLM-L6-v2 understands the meaning of text, even with short queries. The search could still find related results without needing the exact same words, which shows why embeddings are more useful than simple keyword searches.

Adding metadata filtering was also easier than I expected since it mainly works as an extra filter before showing the results. Overall, this activity helped me connect the concepts from embeddings with how retrieval systems and RAG pipelines actually work in practice.

Building the REST API on top of the FilteredVectorStore also made the project feel more real and practical. The chunking part was simple, but it made me realize that real applications probably split text in a smarter way instead of just counting characters. I also liked using Flask’s test client because everything could run directly in the notebook without opening another terminal, which made testing much easier. The metadata validation was  straightforward, but it showed how important it is to keep the data organized before storing it in the vector store.